In [1]:
from pathlib import Path
import shutil

we_dir = Path("NYUS2_2") / "models" / "WeightedEnsembler_L2"
out_dir = Path("NYUS2_2") / "extracted_pkl" / "WeightedEnsembler_L2"
out_dir.mkdir(parents=True, exist_ok=True)

for src in we_dir.rglob("*.pkl"):
    dst = out_dir / src.relative_to(we_dir)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print(f"Copied .pkl files to: {out_dir}")

Copied .pkl files to: NYUS2_2\extracted_pkl\CatBoost_Bag_L1


In [2]:
from pathlib import Path
import pickle

base = Path("NYUS2_2/extracted_pkl/")

pkl_paths = [
    base / "model.pkl",
    base / "utils" / "model_template.pkl",
    base / "utils" / "oof.pkl",
]

def load_pkl(path: Path):
    with path.open("rb") as f:
        return pickle.load(f)

for p in pkl_paths:
    obj = load_pkl(p)
    print("\n===", p.as_posix(), "===")
    print("type:", type(obj))
    # helpful for dict-like objects
    if isinstance(obj, dict):
        print("keys:", list(obj.keys())[:50])
    # helpful for pandas/numpy objects
    for attr in ("shape", "columns", "index"):
        if hasattr(obj, attr):
            try:
                v = getattr(obj, attr)
                print(f"{attr}:", v if attr != "columns" else list(v)[:30])
            except Exception:
                pass


=== NYUS2_2/extracted_pkl/CatBoost_Bag_L1/model.pkl ===
type: <class 'autogluon.core.models.ensemble.stacker_ensemble_model.StackerEnsembleModel'>

=== NYUS2_2/extracted_pkl/CatBoost_Bag_L1/utils/model_template.pkl ===
type: <class 'autogluon.tabular.models.catboost.catboost_model.CatBoostModel'>

=== NYUS2_2/extracted_pkl/CatBoost_Bag_L1/utils/oof.pkl ===
type: <class 'dict'>
keys: ['_oof_pred_proba', '_oof_pred_model_repeats']


In [3]:
from pathlib import Path
import pickle

model_path = Path("NYUS2_2//model.pkl")
with model_path.open("rb") as f:
    m = pickle.load(f)

print("type:", type(m))

# Basic attribute discovery
attrs = [a for a in dir(m) if not a.startswith("_")]
print("public attrs (sample):", attrs[:40])

# What gets pickled (usually the most informative)
state = m.__getstate__() if hasattr(m, "__getstate__") else getattr(m, "__dict__", None)
print("state type:", type(state))
if isinstance(state, dict):
    print("state keys (sample):", list(state.keys())[:60])

# Show likely “interesting” attributes if present
for name in [
    "base_model_names", "model_names", "models",
    "model_weights", "weights", "weights_",
    "params", "params_aux", "metadata",
    "problem_type", "eval_metric", "num_classes",
]:
    if hasattr(m, name):
        val = getattr(m, name)
        print(f"\n{name} -> type={type(val)}")
        try:
            if isinstance(val, dict):
                print("  keys:", list(val.keys())[:50])
            else:
                print(" ", val)
        except Exception as e:
            print("  (could not print)", e)

# List methods that look like summary/importance/fit-related helpers
methods = [n for n in dir(m) if callable(getattr(m, n, None)) and not n.startswith("_")]
print("\nmethods (sample):", [x for x in methods if "weight" in x.lower() or "model" in x.lower()][:40])

FileNotFoundError: [Errno 2] No such file or directory: 'NYUS2_2\\model.pkl'

In [5]:
from pathlib import Path
import pickle
import json

# Use the extracted CatBoost_Bag_L1 folder
base = Path("NYUS2_2/extracted_pkl/CatBoost_Bag_L1")
model_path = base / "model.pkl"

with model_path.open("rb") as f:
    m = pickle.load(f)

# --- build the same "output" as structured data ---
summary = {}
summary["loaded_path"] = model_path.as_posix()
summary["type"] = f"{type(m).__module__}.{type(m).__name__}"

# Basic attribute discovery
attrs = [a for a in dir(m) if not a.startswith("_")]
summary["public_attrs_sample"] = attrs[:40]

# What gets pickled
state = m.__getstate__() if hasattr(m, "__getstate__") else getattr(m, "__dict__", None)
summary["state_type"] = str(type(state))
summary["state_keys_sample"] = list(state.keys())[:60] if isinstance(state, dict) else None

# Show likely “interesting” attributes if present (store as repr to keep it JSON-safe)
interesting_names = [
    "base_model_names", "model_names", "models",
    "model_weights", "weights", "weights_",
    "params", "params_aux", "metadata",
    "problem_type", "eval_metric", "num_classes",
]
summary["interesting_attrs"] = {}
for name in interesting_names:
    if hasattr(m, name):
        val = getattr(m, name)
        entry = {
            "type": f"{type(val).__module__}.{type(val).__name__}",
        }
        if isinstance(val, dict):
            entry["dict_keys_sample"] = list(val.keys())[:50]
        else:
            entry["repr"] = repr(val)
        summary["interesting_attrs"][name] = entry

# List methods that look relevant
methods = [n for n in dir(m) if callable(getattr(m, n, None)) and not n.startswith("_")]
summary["methods_sample"] = [x for x in methods if "weight" in x.lower() or "model" in x.lower()][:40]

# --- write JSON ---
out_path = base / "model_summary.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Wrote:", out_path.as_posix())

FileNotFoundError: [Errno 2] No such file or directory: 'NYUS2_2\\utils\\learner.pkl'

In [8]:
from pathlib import Path
import pickle

learner_path = Path(r"C:\Users\namso\PycharmProjects\JupyterProject\NYUS2_2\predictor.pkl")
print("Trying:", learner_path)
print("Exists?:", learner_path.exists())

with learner_path.open("rb") as f:
    learner = pickle.load(f)

print("Loaded learner type:", type(learner))

# quick peek
public_attrs = [a for a in dir(learner) if not a.startswith("_")]
print("public attrs (sample):", public_attrs[:50])

Trying: C:\Users\namso\PycharmProjects\JupyterProject\NYUS2_2\predictor.pkl
Exists?: True
Loaded learner type: <class 'autogluon.tabular.predictor.predictor.TabularPredictor'>
public attrs (sample): ['Dataset', 'calibrate_decision_threshold', 'can_predict_proba', 'class_labels', 'class_labels_internal', 'class_labels_internal_map', 'classes_', 'clone', 'clone_for_deployment', 'compile', 'decision_threshold', 'delete_models', 'disk_usage', 'disk_usage_per_file', 'distill', 'eval_metric', 'evaluate', 'evaluate_predictions', 'feature_importance', 'feature_metadata', 'feature_metadata_in', 'features', 'fit', 'fit_extra', 'fit_hyperparameters_', 'fit_pseudolabel', 'fit_summary', 'fit_weighted_ensemble', 'has_val', 'info', 'is_fit', 'label', 'leaderboard', 'learning_curves', 'load', 'load_data_internal', 'load_log', 'model_best', 'model_failures', 'model_hyperparameters', 'model_info', 'model_names', 'model_refit_map', 'original_features', 'path', 'persist', 'plot_ensemble_model', 'positive_